#  Flask

- ### Flask es un framework web: Proporciona las herramientas para crear aplicaciones web
- ### Es un microframework: Sin dependencias a librerías externas
- ### Framework ligero

- #### [https://flask.palletsprojects.com/en/stable/](https://flask.palletsprojects.com/en/stable/)
- #### Opciones para el manejo de Bases de Datos
    - #### Adaptadores de base de datos PostgreSQL como Psycopg
        - #### [https://pypi.org/project/psycopg2](https://pypi.org/project/psycopg2)
    - #### Framework ORM (Object Relational Mapper) como SQLAlchemy
        - #### [https://www.sqlalchemy.org](https://www.sqlalchemy.org)


# Instalar Flask
- ### Desde la terminal en el ambiente (TPS), instalar flask con pip 

In [ ]:
pip install flask

### Comprobar la versión instalada de Flask

In [ ]:
import flask
print(flask.__version__)

In [2]:
# Crear el directorio de la aplicación
!mkdir app_web


## Aplicacion Web en FLASK 

- ### Una vez que cree la instancia app, se utiliza para gestionar las solicitudes web entrantes y enviar respuestas al usuario. 

- ###  @app.route es un decorador que convierte una función Python regular en una función vista de Flask

- ###  Convierte el valor de devolución de la función en una respuesta HTTP que se mostrará mediante un cliente HTTP

- ###   Pasa el valor '/' a @app.route() para indicar que esta función responderá a las solicitudes web para la URL /, que es la URL principal.

- ###  La función de vista hello() devuelve la cadena 'Hello, World!'​​ como respuesta.


In [ ]:
# Crear el archivo inicio.py

from flask import Flask

app = Flask(__name__)


@app.route('/')
def inicio():
    return '¡Hola, mundo! Esta es mi primera aplicación Flask.'

if __name__ == '__main__':
    app.run(debug=True)


### Ejecutar la aplicacion


#### Desde la terminal, en el directorio de la aplicación

#### <span style="color:#0485CF"> python inicio.py </span>



```
* Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with stat
 * Debugger is active!
 * Debugger PIN: 947-720-893
127.0.0.1 - - [16/Feb/2024 14:56:02] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [16/Feb/2024 14:56:02] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [16/Feb/2024 14:58:34] "GET / HTTP/1.1" 200 -
```



#### Otra forma de ejecuta la aplicación web usando el puerto 5000
#### <span style="color:#0485CF"> flask --app inicio.py --debug run -p 5000 </span>



# Entrar al navegador para ver los resultados
## http://127.0.0.1:5000


# Ejemplo de página de login con HTML

### 1. Crear la carpeta APP_WEB2, como directorio de aplicación
### 2. Crear la carpeta "templates" en el directorio de la aplicación
### 3. Crear una plantilla HTML la página de inicio de sesión (login.html)


# templates/login.html:

In [ ]:

<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Iniciar sesión</title>
</head>
<body>
    <h1>Iniciar sesión</h1>
    <form action="/login" method="post">
        <label for="username">Usuario:</label><br>
        <input type="text" id="usuario" name="usuario"><br>
        <label for="password">Contraseña:</label><br>
        <input type="password" id="contrasena" name="contrasena"><br><br>
        <input type="submit" value="Iniciar sesión">
    </form>
</body>
</html>

### 4. En la raiz del directorio de la aplicación (APP_WEB2) crear un archivo con la aplicación Flask,  definir una ruta para la página de inicio de sesión y otra para manejar el envío del formulario de inicio de sesión.

In [ ]:
# Archivo inicio.py
from flask import Flask, render_template, request, redirect, url_for


app = Flask(__name__)

# Datos de usuarios (simulados para el ejemplo)

USUARIOS = {
    'usuario1': 'u1',
    'usuario2': 'u2'
}



@app.route('/inicio')
def inicio():
    return '¡Hola, mundo! Esta es mi primera aplicación Flask.'

@app.route('/')
@app.route('/login', methods=['GET', 'POST'])
def login():
    if request.method == 'POST':
        username = request.form.get('usuario')
        password = request.form.get('contrasena')
        if username in USUARIOS and USUARIOS[username] == password:
            # Iniciar sesión exitosa, redirigir a otra página
            return redirect(url_for('inicio'))
        else:
            # Credenciales incorrectas, mostrar mensaje de error
            return 'Credenciales incorrectas. <a href="/login">Intenta de nuevo</a>'
    else:
        return render_template('login.html')
    


if __name__ == '__main__':
    app.run(debug=True)


# Entrar al navegador para ver los resultados
## http://127.0.0.1:5000/login


## Conectando con la Base de Datos

### 1. Crear la carpeta APP_WEB3, como directorio de aplicación
### 2. En el directorio raíz de la aplicación, crear el archivo con la clase para conectarse a la base de datos: postgres_db.py
### 3. En el directorio raíz de la aplicación, crear el archivo con la aplicación de flask: inicio.py



In [1]:

#archivo postgres_db.py
from psycopg2.pool import ThreadedConnectionPool
from contextlib import contextmanager
import atexit

db_config = { "host" : "localhost",
                "database" : "sistema_abc",
                "user" : "uacm",
                "password" : "uacm1"}


class PostgresDB:
    def __init__(self):
        self.app = None
        self.pool = None

    def init_app(self, app):
        self.app = app
        self.connect()
        # Se ejecuta automáticamente cuando el proceso Python termina
        # Cierra las conexiones del pool
        atexit.register(self.close)

    def connect(self):
        # Se crean un pool de conexiones a la base de datos
        self.pool = ThreadedConnectionPool(minconn=1, maxconn=30, **db_config)


    # El decorador @contextmanager, se utiliza para crear administradores de contexto (context managers), 
    # convierte una función generadora en un context manager, es decir, 
    # algo que puedes usar con with.
    @contextmanager
    def get_cursor(self):
        if self.pool is None:
            self.connect()
        # Obtiene una conexión del pool
        con = self.pool.getconn()
        cur = con.cursor()
        try:
            # "pausa" y entrega el cursor al bloque with
            yield cur
            # Al salir del with
            # Se ejecuta si no hubo excepciones
            con.commit()
        except Exception:
            con.rollback()  # cancelar los cambios
            # Vuelve a lanzar la misma excepción que se capturó
            raise
        finally:
            cur.close()          # cerrar el cursor
            self.pool.putconn(con)  # devolver conexión al pool

    def close(self):
        if self.pool:
            self.pool.closeall()  # cierra todas las conexiones del pool


# Instancia a la DB

pgdb = PostgresDB()            


In [ ]:
# inicio.py
import atexit
from flask import Flask, jsonify
from postgres_db import pgdb   # crea la instancia

def create_app():
    app = Flask(__name__)
    pgdb.init_app(app)
    return app

app = create_app()

@app.route('/consulta')
def consultaBD():
    resultados = []
    try:
        with pgdb.get_cursor() as cur:
            cur.execute("SELECT * FROM productos")
            filas = cur.fetchall()
            for fila in filas:
                print(fila)
                resultados.append(str(fila))
    except Exception as e:
        print("Error:", e)
        return f"Error: {e}", 500

    return "Consulta realizada.\n" + " ".join(resultados)


if __name__ == '__main__':
    app.run(debug=True)

# Entrar al navegador para ver los resultados
## http://127.0.0.1:5000/consulta


### Ejemplo de Consulta y despliegue de los datos en una página HTML

### 1. Crear la carpeta APP_WEB4, como directorio de aplicación
### 2. En el directorio raíz de la aplicación, crear el archivo con la clase para conectarse a la base de datos: postgres_db.py
### 3. En el directorio raíz de la aplicación, crear el archivo con la aplicación de flask: inicio.py
### 4. En el directorio raíz de la aplicación, crear la carpeta "templates"
### 4. Dentro de la carpeta "templates" crear los archivos:
- ### Página de login (login.html)
- ### Página del menú de la aplicación (menu.html)
- ### Página de consulta de datos (consulta.html)




### Crear el archivo dentro de la carpeta: templates/login.html

In [ ]:
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Iniciar sesión</title>
</head>
<body>
    <h1>Iniciar sesión</h1>
    <form action="/login" method="post">
        <label for="username">Usuario:</label><br>
        <input type="text" id="usuario" name="usuario"><br>
        <label for="password">Contraseña:</label><br>
        <input type="password" id="contrasena" name="contrasena"><br><br>
        <input type="submit" value="Iniciar sesión">
    </form>
</body>
</html>

### Crear Archivo dentro de la carpeta: templates/menu.html

In [ ]:
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <title>Página de Inicio</title>
</head>
<body>
    <h1>Bienvenido al menú de  aplicación</h1>
    <a href="/consulta">Consultar datos</a>
</body>
</html>

### Crear archivo: inicio.py (el archivo se debe colocar en el directorio de la appliación web)

In [ ]:
from flask import Flask, render_template, request, redirect, url_for


import atexit
from flask import Flask, jsonify
from postgres_db import pgdb   # crea la instancia

# Datos de usuarios (simulados para el ejemplo)

USUARIOS = {
    'usuario1': 'u1',
    'usuario2': 'u2'
}


def create_app():
    app = Flask(__name__)
    pgdb.init_app(app)
    atexit.register(pgdb.close)
    return app

app = create_app()

#--------------------------------------------------
@app.route('/menu', methods=['GET', 'POST'])
def menu():
        return render_template('menu.html')


#--------------------------------------------------

@app.route('/login', methods=['GET', 'POST'])
def login():
    if request.method == 'POST':
        username = request.form.get('usuario')
        password = request.form.get('contrasena')
        if username in USUARIOS and USUARIOS[username] == password:
            # Iniciar sesión exitosa, redirigir a otra página
            return redirect(url_for('menu'))
        else:
            # Credenciales incorrectas, mostrar mensaje de error
            return 'Credenciales incorrectas. <a href="/login">Intenta de nuevo</a>'
    else:
        return render_template('login.html')
    

#--------------------------------------------------

@app.route('/consulta', methods=['GET', 'POST'])
def consultaBD_html():
    resultados = []
    try:
        with pgdb.get_cursor() as cur:
            cur.execute("SELECT * FROM productos")
            resultados = cur.fetchall()
    except Exception as e:
        print("Error:", e)

    return render_template('consulta.html', datos=resultados)


#--------------------------------------------------



if __name__ == '__main__':
    app.run(debug=True)

### 2. Crear el archivo dentro de la carpeta templates/consulta.html, que es la página que desplegará los datos

- #### En Flask, la etiqueta {% for %} se utiliza para crear bucles for en plantillas.
- #### Las variables se usan con {{ }},


In [ ]:
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Consulta</title>
    
</head>
<body>
    <h1>Lista de Datos</h1>
    <table>
        <tr>
            <th>id</th>
            <th>DESC</th>
            <th>precio</th>
        </tr>
        {% for id, desc, precio in datos %}
        <tr>
            <td>{{ id }}</td>
            <td>{{ desc }}</td>
            <td>{{ precio }}</td>
        </tr>
        {% endfor %}
    </table>
</body>
</html>


# Entrar al navegador para ver los resultados
## http://127.0.0.1:5000/login


# Ejercicio

- #### 1. Hacer una aplicación web que contenga un login de inicio de sesión
- #### 2. Tenga un menú donde se desplieguen las opciones del sistema
- #### 3. Consulte los datos de la tabla Empleados (todos)
- #### 4. Consulte los datos de la tabla Empleados menores a cierto monto


